In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if (root / "src").exists():
    sys.path.append(str(root / "src"))
elif (root.parent / "src").exists():
    sys.path.append(str(root.parent / "src"))

from collections import Counter
from dataset import load_dataset, count_samples, get_label_mapping
import numpy as np

X, y = load_dataset()
sample_count = count_samples()
label_mapping = get_label_mapping()
number_of_classes = len(label_mapping)
files_per_class = {name: Counter(y)[idx] for name, idx in label_mapping.items()}
_first = np.load(X[0], allow_pickle=True)
first_sample_shape = _first.shape
first_sample_dtype = _first.dtype

def format_files_per_class(items):
    return {name: items[name] for name in sorted(items)}

print("Number of classes:", number_of_classes)
print("Number of files per class:", format_files_per_class(files_per_class))
print("Shape of the first sample:", first_sample_shape)
print("Data type:", first_sample_dtype)
print("Total number of samples:", sample_count)

Number of classes: 4
Number of files per class: {'ADS_B': 160, 'FM_broadcast': 160, 'ISM_sensors': 160, 'noise': 160}
Shape of the first sample: ()
Data type: object
Total number of samples: 640


In [2]:
from model_2d.preprocess_2d import summarize_dataset

summary = summarize_dataset(X, y, batch_size=8, hop_length=2048)

print(f"Loaded {summary['loaded_samples']} samples")
print(f"\nOriginal sample shape: {summary['original_sample_shape']}")
print(f"Shape after preprocessing: {summary['preprocessed_sample_shape']}")
print(f"Data type: {summary['data_type']}")
print(f"\nTrain: {summary['train']}")
print(f"Validation: {summary['validation']}")
print(f"Test: {summary['test']}")
print(f"\nBatch shape: {summary['batch_shape']}")
print("\nTraining class distribution:")
for label, count in summary["train_class_distribution"].items():
    print(f"Label {label}: {count} samples")
print(f"\nExample spectrograms saved to: {summary['example_spectrograms_dir']}")


Loaded 640 samples

Original sample shape: (512000,)
Shape after preprocessing: (1, 128, 128)
Data type: torch.float32

Train: 110900
Validation: 16167
Test: 31443

Batch shape: torch.Size([8, 1, 128, 128])

Training class distribution:
Label 0: 28328 samples
Label 1: 26593 samples
Label 2: 29043 samples
Label 3: 26936 samples

Example spectrograms saved to: C:\Users\ishas\Downloads\ISSA\results\2d\spectrograms


In [3]:
import torch

from model_2d.model_2d import SpectrogramCNN


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SpectrogramCNN(num_classes=5).to(device)
dummy_input = torch.randn(1, 1, 128, 128, device=device)
output = model(dummy_input)
predicted_class = torch.argmax(output, dim=1)

print("Model architecture:")
print(model)
print()
print("Number of trainable parameters:", count_trainable_parameters(model))
print("Device:", device)
print("Dummy input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(output.shape))
print("Output values:")
print(output)
print("Predicted class:", predicted_class.item())
print("Any NaNs in input:", torch.isnan(dummy_input).any().item())
print("Any NaNs in output:", torch.isnan(output).any().item())

Model architecture:
SpectrogramCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

In [4]:
from model_2d.train_2d import main, BEST_MODEL_PATH

model, history = main(num_epochs=20, hop_length=2048)

last_epoch = len(history["train_loss"])

print(f"Train Loss: {history['train_loss'][-1]:.2f}")
print(f"Train Accuracy: {history['train_accuracy'][-1] * 100:.1f}%")
print(f"Validation Loss: {history['validation_loss'][-1]:.2f}")
print(f"Validation Accuracy: {history['validation_accuracy'][-1] * 100:.1f}%")
print()
print(f"Best Validation Accuracy: {history['best_validation_accuracy'] * 100:.1f}%")
print(f"Model saved to {BEST_MODEL_PATH}")

Indexing windows...
Indexed 110900 windows.
Indexing windows...
Indexed 16167 windows.
Indexing windows...
Indexed 31443 windows.
Epoch 1/20 — train_loss=0.4190 train_acc=0.8246 val_loss=0.2480 val_acc=0.8911
Epoch 2/20 — train_loss=0.2726 train_acc=0.8851 val_loss=0.2118 val_acc=0.9239
Epoch 3/20 — train_loss=0.2350 train_acc=0.9011 val_loss=0.1888 val_acc=0.9082
Epoch 4/20 — train_loss=0.2127 train_acc=0.9105 val_loss=0.1773 val_acc=0.9260
Epoch 5/20 — train_loss=0.1966 train_acc=0.9170 val_loss=0.1426 val_acc=0.9465
Epoch 6/20 — train_loss=0.1830 train_acc=0.9224 val_loss=0.1454 val_acc=0.9455
Epoch 7/20 — train_loss=0.1742 train_acc=0.9267 val_loss=0.1342 val_acc=0.9376
Epoch 8/20 — train_loss=0.1678 train_acc=0.9292 val_loss=0.1269 val_acc=0.9497
Epoch 9/20 — train_loss=0.1604 train_acc=0.9307 val_loss=0.1305 val_acc=0.9391
Epoch 10/20 — train_loss=0.1543 train_acc=0.9339 val_loss=0.1531 val_acc=0.9214
Epoch 11/20 — train_loss=0.1482 train_acc=0.9363 val_loss=0.1306 val_acc=0.9490

In [5]:
from model_2d.evaluate_2d import evaluate, BEST_MODEL_PATH

results = evaluate(BEST_MODEL_PATH, hop_length=2048)

print(f"Test Loss: {results['test_loss']:.2f}")
print(f"Test Accuracy: {results['test_accuracy'] * 100:.1f}%")
print(f"Precision: {results['macro_precision'] * 100:.1f}%")
print(f"Recall: {results['macro_recall'] * 100:.1f}%")
print(f"F1-score: {results['macro_f1'] * 100:.1f}%")
print()
print("Classification report saved to results/2d/classification_report.txt")
print("Confusion matrix saved to results/2d/confusion_matrix/confusion_matrix.png")
print("Evaluation metrics saved to results/2d/evaluation.json")

Indexing windows...
Indexed 110900 windows.
Indexing windows...
Indexed 16167 windows.
Indexing windows...
Indexed 31443 windows.
Test Loss: 0.16
Test Accuracy: 93.2%
Precision: 93.7%
Recall: 93.1%
F1-score: 93.1%

Classification report saved to results/2d/classification_report.txt
Confusion matrix saved to results/2d/confusion_matrix/confusion_matrix.png
Evaluation metrics saved to results/2d/evaluation.json


In [6]:
from model_2d.predict_2d import predict_path,print_prediction

results = predict_path()

for index, result in enumerate(results):
    if index > 0:
        print()
    print_prediction(result)

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\ADS_B.npy

Predicted Class:
ADS_B

Confidence:
95.9%

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\FM_broadcast_20260126_023055_711669.npy

Predicted Class:
FM_broadcast

Confidence:
95.0%

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\ism.npy

Predicted Class:
ISM_sensors

Confidence:
98.4%

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\live.npy

Predicted Class:
FM_broadcast

Confidence:
99.8%
